In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import io

In [ ]:
diabetes = pd.read_csv(
    "filterzoo7.csv",
    header=0)
diabetes.head()

FileNotFoundError: [Errno 2] No such file or directory: 'filterzoo7.csv'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
labels = diabetes['認養可能性']             # 目標變數 (0/1)
x_data = diabetes.drop(['認養可能性','animal_id','animal_createtime'], axis=1)  # 8 個自變數
print(x_data)

In [ ]:
from sklearn.model_selection import train_test_split
# train_test_split 參數說明：
#   test_size      : 測試集比例，0.33 表示 33% 作為測試集
#   random_state   : 亂數種子，固定值可使結果可重現
#   stratify=labels: 若資料類別不平衡，可加上以維持各類比例
X_train, X_test, y_train, y_test = train_test_split(
    np.asanyarray(x_data), np.asanyarray(labels),
    test_size=0.33, random_state=101)

print('Train:', X_train.shape, 'Test:', X_test.shape)
print('測試集類別分佈:')
print(pd.Series(y_test).value_counts())

In [ ]:
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, matthews_corrcoef, confusion_matrix,
                             classification_report)


# =============================================================================
# 整體考量（對應簡報第 44 / 47 頁）：分類常用指標
# =============================================================================
def clf_eval(y_true, y_pred, label=''):
    """計算分類整體考量指標。"""
    acc = accuracy_score(y_true, y_pred)
    pre = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred,    zero_division=0)
    f1  = f1_score(y_true, y_pred,        zero_division=0)
    try:
        mcc = matthews_corrcoef(y_true, y_pred)
    except Exception:
        mcc = float('nan')
    print('[%s] Acc=%.4f  P=%.4f  R=%.4f  F1=%.4f  MCC=%.4f' %
          (label, acc, pre, rec, f1, mcc))
    return acc, pre, rec, f1, mcc


# =============================================================================
# 實務考量（對應簡報第 45 頁分類版本）：機率預測誤差 < 門檻 的通過率
# =============================================================================
def threshold_pass_rate_clf(y_true, y_prob, threshold=0.30, label=''):
    """分類實務考量：誤差 = |預測機率 − 實際 0/1|；正確率 = (誤差 < 門檻) 比例。"""
    y_true = np.asarray(y_true).ravel()
    y_prob = np.asarray(y_prob).ravel()
    err = np.abs(y_prob - y_true)
    count = int(np.sum(err < threshold))
    rate = count / len(err)
    print('[%s] 門檻=%.2f → 通過 %d/%d 筆，正確率 %.2f%%' %
          (label, threshold, count, len(err), rate * 100))
    return count, rate


# =============================================================================
# 統一評估介面：對「訓練集」與「測試集」皆同步計算「整體考量」與「實務考量」
# =============================================================================
def evaluate_classification(model_name, y_train_true, y_train_pred,
                            y_test_true, y_test_pred,
                            y_train_prob=None, y_test_prob=None,
                            threshold=0.30, show_cm=True):
    """同步輸出訓練集與測試集的整體考量 + 實務考量。"""
    print('=' * 65)
    print('模型：%s' % model_name)
    print('-' * 65)
    print('[整體考量]')
    clf_eval(y_train_true, y_train_pred, '%s (train)' % model_name)
    clf_eval(y_test_true,  y_test_pred,  '%s (test) ' % model_name)
    if y_train_prob is not None and y_test_prob is not None:
        print('[實務考量]（機率預測誤差 < 門檻 → 視為通過）')
        threshold_pass_rate_clf(y_train_true, y_train_prob, threshold, '%s (train)' % model_name)
        threshold_pass_rate_clf(y_test_true,  y_test_prob,  threshold, '%s (test) ' % model_name)
    if show_cm:
        print('[測試集混淆矩陣]')
        print(confusion_matrix(y_test_true, y_test_pred))
    print('=' * 65)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
import numpy as np

# Convert X_train and X_test categorical features to numerical using OneHotEncoder
# It's important to fit the encoder only on X_train to prevent data leakage.
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

X_train_encoded = encoder.fit_transform(X_train)
X_test_encoded = encoder.transform(X_test)

# Convert y_train and y_test categorical labels to numerical using LabelEncoder
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

Lrn = LogisticRegression(
    random_state=0,                 # 亂數種子，確保結果可重現
    max_iter=1000)                  # 最大迭代次數（預設 100 太少，提高為 1000）
Lrn.fit(X_train_encoded, y_train_encoded)
# 同步取得訓練與測試的預測標籤與機率
y_pred_tr  = Lrn.predict(X_train_encoded)
y_pred_te  = Lrn.predict(X_test_encoded)
y_prob_tr  = Lrn.predict_proba(X_train_encoded)[:, 1]
y_prob_te  = Lrn.predict_proba(X_test_encoded)[:, 1]
# 對應簡報「整體考量」+「實務考量」，同步輸出訓練/測試準確性
evaluate_classification('LogisticRegression', y_train_encoded, y_pred_tr, y_test_encoded, y_pred_te,
                        y_prob_tr, y_prob_te, threshold=0.30)


In [ ]:
from sklearn.neural_network import MLPClassifier

model = MLPClassifier(
    hidden_layer_sizes=(6, 2),  # 兩層隱藏層；簡報建議 (64,32) 為平衡架構
    activation='relu',            # 激勵函數：ReLU 緩解梯度消失，計算簡單
    solver='adam',                # Adam 結合動量與自適應學習率，適合多數情況
    batch_size='auto',            # 'auto' = min(200, n_samples)
    learning_rate='constant',     # 學習率策略：'constant'/'invscaling'/'adaptive'
    learning_rate_init=0.001,     # 初始學習率（adam/sgd 使用）
    power_t=0.5,                  # solver='sgd' 時的動態學習率指數
    max_iter=500,                 # 最大迭代次數
    shuffle=True,                 # 每個 epoch 是否打亂訓練資料
    random_state=1,               # 亂數種子
    momentum=0.9,                 # 動量項（僅 sgd 有效）
    alpha=0.0001)                 # L2 正則化強度，越大越能抑制過擬合

In [ ]:
model.fit(X_train, y_train)
y_pred_tr = model.predict(X_train);  y_pred_te = model.predict(X_test)
# MLPClassifier 預設可呼叫 predict_proba
y_prob_tr = model.predict_proba(X_train)[:, 1]
y_prob_te = model.predict_proba(X_test)[:, 1]
evaluate_classification('MLP (64,32)', y_train, y_pred_tr, y_test, y_pred_te,
                        y_prob_tr, y_prob_te, threshold=0.30)

In [ ]:
from sklearn import svm
from sklearn.preprocessing import StandardScaler

# 簡報建議：SVM 使用前先做特徵標準化（避免大尺度特徵主導距離）
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)  # fit + transform
X_test_s  = scaler.transform(X_test)        # 僅 transform（使用訓練集的 mean/std）

In [ ]:
clf = svm.SVC(
    kernel='rbf',                   # 徑向基核（RBF）：最常用、可處理非線性
    C=1.0,                          # 懲罰係數
    gamma='scale',                  # gamma='scale' = 1/(n_features * X.var())
    probability=True)               # 啟用機率輸出
clf.fit(X_train_s, y_train)
y_pred_tr = clf.predict(X_train_s);  y_pred_te = clf.predict(X_test_s)
y_prob_tr = clf.predict_proba(X_train_s)[:, 1]
y_prob_te = clf.predict_proba(X_test_s)[:, 1]
evaluate_classification('SVC rbf', y_train, y_pred_tr, y_test, y_pred_te,
                        y_prob_tr, y_prob_te, threshold=0.30)

In [ ]:
from sklearn.naive_bayes import CategoricalNB
from sklearn.preprocessing import KBinsDiscretizer

# 將連續特徵離散化為類別
disc = KBinsDiscretizer(
    n_bins=100,                       # 分成 5 個區間（bins）
    encode='ordinal',               # 編碼方式：'ordinal'=整數，'onehot'=獨熱
    strategy='uniform')             # 切分策略：'uniform'=等寬，'quantile'=等頻
X_train_d = disc.fit_transform(X_train)
X_test_d  = disc.transform(X_test)

cnb = CategoricalNB(
    alpha=1.0)                      # Laplace 平滑（同 Multinomial）
cnb.fit(X_train_d, y_train)
y_pred_tr = cnb.predict(X_train_d);  y_pred_te = cnb.predict(X_test_d)
y_prob_tr = cnb.predict_proba(X_train_d)[:, 1]
y_prob_te = cnb.predict_proba(X_test_d)[:, 1]
evaluate_classification('CategoricalNB', y_train, y_pred_tr, y_test, y_pred_te,
                        y_prob_tr, y_prob_te, threshold=0.30)

In [ ]:
import joblib

# 假設你最後決定用 MLP 模型（因為在你的測試中它達到 99.9% 的準確度！）
# 先確保模型已經 fit 過了：model.fit(X_train, y_train)

# 儲存模型
joblib.dump(model, 'mlp_model.pkl')
print("模型儲存成功！檔案名稱為 mlp_model.pkl")

In [ ]:
import joblib

# 這裡的 model 必須是你前面跑過 model.fit(X_train, y_train) 的那個變數名稱
# 執行這段會在你目前的目錄下產生一個 mlp_model.pkl 檔案
joblib.dump(model, 'mlp_model.pkl2')
print("模型儲存成功！")

In [ ]:
import joblib
from sklearn.neural_network import MLPClassifier

# 1. 建立模型
model = MLPClassifier(
    hidden_layer_sizes=(6, 2),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=1,
    alpha=0.0001
)

# 2. 訓練模型（這一步絕對不能漏掉！！）
# 必須傳入你的訓練資料，讓模型學習
model.fit(X_train, y_train)

# 3. 儲存「已經訓練好」的模型
joblib.dump(model, 'mlp_model.pkl')
print("🎉 這次是真的把『訓練好』的模型儲存成功囉！")